# Análisis de errores del segmentador en la base de la Table 2 (etapa 1, sin GPU)
Especificación: `claude/encargo_errores_segmentacion_v3.md`. Reemplaza el análisis del 29-oct (`errores-segmentacion` v2), que usaba 576 px, emparejamiento húngaro y otro umbral de área.

**Settings:** CPU, Internet OFF. Sin re-entrenar ni re-inferir.
**Inputs:**
- `map-segmentadores`: `instancias.pkl.gz` (matriz de confusión de píxeles por foto y modelo, resolución nativa; YOLO pintado con conf ≥ 0,25);
- `datasets-limpios-icdas-segmentacion`: `segmentation_manifest.csv` y `fdi_segmentation_curated.zip` (máscaras; los colores fuera del labelmap son dientes temporales);
- `f1-iou05`: `f1_iou05_resumen.csv` (chequeo previo);
- `benchmark-segmentacion-cv-nino`: `results/<modelo>/fold<k>_preds.npz` de SegFormer-B2 y YOLO (solo para el cruce píxel a píxel con los temporales, punto 4).

**Definiciones (por modelo y foto):** R = clases FDI de la referencia (ng > 0); P = clases predichas con ≥ 50 px; IoU(c,d) = cm[c,d] / (ng_c + npx_d − cm[c,d]).
- Identificado: IoU(c,c) > 0,5 (definición del paper).
- Diente de referencia no identificado → 1) **número equivocado** (existe d ≠ c en P con IoU(c,d) > 0,5); 2) **ubicado, contorno insuficiente** (no 1 y píxeles de diente predichos de cualquier clase cubren ≥ 50 % del diente); 3) **no detectado** (cobertura < 50 %).
- Clase predicha no identificada (d en P sin IoU(d,d) > 0,5) → **contraparte de número equivocado**; **sobre fondo** (≥ 50 % de sus píxeles en fondo de la referencia); **fragmento/contorno insuficiente** (el resto).

Bootstrap de niños: 1.000 remuestreos, `default_rng(0)`, `rng.choice(KIDS, len(KIDS))` (los mismos de `metricas_ic95.csv` y `f1-iou05`).

In [ ]:
import os, glob, gzip, pickle, time, zipfile, re
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 200)
IN, W, TMP = '/kaggle/input', '/kaggle/working', '/kaggle/tmp'
OUT = f'{W}/errores_v3'; os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
MODELS = ['segformer_b2', 'yolo11m_seg', 'deeplabv3p_r50', 'unetpp_r50']
LABEL = {'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg', 'deeplabv3p_r50': 'DeepLabV3+', 'unetpp_r50': 'U-Net++'}
PIX_MODELS = ['segformer_b2', 'yolo11m_seg']
MIN_PX, TOL, N_REF, B = 50, 0.001, 9710, 1000
N_MIX_IMG, N_MIX_KIDS = 35, 10
CAT_REF = ['identified', 'wrong_number', 'insufficient_contour', 'not_detected']
CAT_PRED = ['wrong_number_counterpart', 'on_background', 'fragment']
T0 = time.time()
def find(name):
    h = sorted(glob.glob(f'{IN}/**/{name}', recursive=True)); return h[0] if h else None

## Manifiesto, labelmap, máscaras y dientes temporales
Se leen las máscaras originales solo para marcar los píxeles con colores fuera del labelmap (dientes temporales). Dentición mixta = foto con ≥ 50 px de esos colores (se compara con las 35 fotos de 10 niños del 29-oct).

In [ ]:
man = pd.read_csv(find('segmentation_manifest.csv'))
DATA = f'{TMP}/seg'
if not glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True):
    zipfile.ZipFile(find('fdi_segmentation_curated.zip')).extractall(DATA)
ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA}/**/{man.mask_file[0]}', recursive=True)[0]))
LM = (glob.glob(f'{DATA}/**/labelmap.txt', recursive=True) or [find('labelmap.txt')])[0]
lines = [l.strip() for l in open(LM, encoding='utf-8') if l.strip() and not l.startswith('#')]
NAMES, COLORS = [], []
for l in lines:
    p = l.split(':'); NAMES.append(p[0].strip()); COLORS.append(tuple(int(v) for v in p[1].split(',')) if len(p) > 1 and p[1] else None)
bg = [i for i, n in enumerate(NAMES) if n.lower() in ('background', 'fondo', 'bg')]
if bg and bg[0] != 0:
    i = bg[0]; NAMES.insert(0, NAMES.pop(i)); COLORS.insert(0, COLORS.pop(i))
NC = len(NAMES)
def to_fdi(u):
    if u is None or not 1 <= u <= 32: return None
    if u <= 8: return 10 + (9 - u)
    if u <= 16: return 20 + (u - 8)
    if u <= 24: return 30 + (25 - u)
    return 40 + (u - 24)
FDI = [None] + [to_fdi(int(re.search(r'\d+', n).group())) for n in NAMES[1:]]
assert NC == 33 and all(f is not None for f in FDI[1:])
TEETH = list(range(1, NC))
m0 = cv2.imread(os.path.join(ROOT, man.mask_file[0]), cv2.IMREAD_UNCHANGED); RGB_MASK = m0.ndim == 3
KNOWN = np.array(sorted({0} | {(c[0] << 16) | (c[1] << 8) | c[2] for c in COLORS if c is not None}), np.int64)
def read_info(rel):
    m = cv2.imread(os.path.join(ROOT, rel), cv2.IMREAD_UNCHANGED)
    if RGB_MASK:
        m = cv2.cvtColor(m[..., :3], cv2.COLOR_BGR2RGB)
        key = (m[..., 0].astype(np.int64) << 16) | (m[..., 1].astype(np.int64) << 8) | m[..., 2]
        prim = ~np.isin(key, KNOWN)
    else:
        key = m.astype(np.int64); prim = key >= NC
    cols = {}
    if prim.any():
        u, n = np.unique(key[prim], return_counts=True); cols = dict(zip(u.tolist(), n.tolist()))
    return int(prim.sum()), (np.packbits(prim.ravel()) if prim.any() else None), prim.shape, cols
with ThreadPoolExecutor(8) as ex:
    INFO = dict(zip(man.image_id, ex.map(read_info, man.mask_file)))
man['prim_px'] = man.image_id.map(lambda j: INFO[j][0])
man['mixed'] = man.prim_px >= MIN_PX
assert man.groupby('child_id').fold.nunique().max() == 1
allcols = {}
for j in man.image_id:
    for k, v in INFO[j][3].items(): allcols[k] = allcols.get(k, 0) + v
print(len(man), 'fotos |', man.child_id.nunique(), 'niños | fotos con algún píxel fuera del labelmap:', int((man.prim_px > 0).sum()),
      '| dentición mixta (≥ 50 px):', int(man.mixed.sum()), 'fotos de', man[man.mixed].child_id.nunique(), 'niños')
print('colores fuera del labelmap (RGB: píxeles):', {f'{k >> 16},{(k >> 8) & 255},{k & 255}': v for k, v in sorted(allcols.items(), key=lambda x: -x[1])})
old = [p for p in glob.glob(f'{IN}/**/*.csv', recursive=True) if re.search(r'mixt|errores', os.path.basename(p), re.I) and 'errores_v3' not in p]
print('salidas del 29-oct encontradas para comparar:', old)
for p in old:
    try:
        o = pd.read_csv(p); col = [c for c in o.columns if 'image' in c.lower()]
        if col and len(o) < 200:
            s_old = set(o[col[0]].astype(str)); s_new = set(man[man.mixed].image_id.astype(str))
            print(f'  {os.path.basename(p)}: {len(s_old)} fotos; coinciden {len(s_old & s_new)}; solo antes {len(s_old - s_new)}; solo ahora {len(s_new - s_old)}')
    except Exception as e: print('  no se pudo leer', p, e)
if int(man.mixed.sum()) != N_MIX_IMG or man[man.mixed].child_id.nunique() != N_MIX_KIDS:
    print(f'AVISO: dentición mixta distinta de {N_MIX_IMG} fotos / {N_MIX_KIDS} niños del 29-oct (se informa, no se detiene)')

## Matrices de confusión del caché y clasificación de errores

In [ ]:
INST = pickle.load(gzip.open(find('instancias.pkl.gz')))
CM = {}
for m in MODELS:
    assert len(INST[m]) == len(man) and all('cm' in INST[m][j] for j in man.image_id), f'falta la matriz de confusión de {m}'
    CM[m] = {j: INST[m][j]['cm'].astype(np.int64) for j in man.image_id}
del INST
assert next(iter(CM[MODELS[0]].values())).shape == (NC, NC)
rows_ref, rows_pred = [], []
for m in MODELS:
    for j, f, child, view, cond, mix in zip(man.image_id, man.fold, man.child_id, man.view, man.condition, man.mixed):
        cm = CM[m][j]; ng = cm.sum(1); npx = cm.sum(0)
        iou = cm / np.maximum(1, ng[:, None] + npx[None, :] - cm)
        P = [d for d in TEETH if npx[d] >= MIN_PX]; R = [c for c in TEETH if ng[c] > 0]
        for c in R:
            wd = None
            if iou[c, c] > 0.5: cat = 'identified'
            else:
                cand = [d for d in P if d != c and iou[c, d] > 0.5]
                if cand: cat, wd = 'wrong_number', cand[0]
                elif cm[c, 1:].sum() / ng[c] >= 0.5: cat = 'insufficient_contour'
                else: cat = 'not_detected'
            rows_ref.append((LABEL[m], j, f, child, view, cond, mix, FDI[c], int(ng[c]), iou[c, c], cm[c, 1:].sum() / ng[c], cat, FDI[wd] if wd else np.nan))
        for d in P:
            if iou[d, d] > 0.5: continue
            if any(c != d and iou[c, d] > 0.5 for c in R): cat = 'wrong_number_counterpart'
            elif cm[0, d] / npx[d] >= 0.5: cat = 'on_background'
            else: cat = 'fragment'
            rows_pred.append((LABEL[m], j, f, child, view, cond, mix, FDI[d], int(npx[d]), cm[0, d] / npx[d], cat))
        rows_pred += [(LABEL[m], j, f, child, view, cond, mix, FDI[d], int(npx[d]), cm[0, d] / npx[d], 'tp') for d in P if iou[d, d] > 0.5]
COLS = ['model', 'image_id', 'fold', 'child_id', 'view', 'condition', 'mixed']
TR = pd.DataFrame(rows_ref, columns=COLS + ['tooth', 'gt_px', 'iou', 'coverage', 'category', 'pred_tooth'])
TP = pd.DataFrame(rows_pred, columns=COLS + ['tooth', 'pred_px', 'frac_bg', 'category'])
TR.to_csv(f'{OUT}/errores_v3_dientes_referencia.csv.gz', index=False); TP.to_csv(f'{OUT}/errores_v3_clases_predichas.csv.gz', index=False)
print(TR.groupby('model').size().to_string()); print(pd.crosstab(TR.model, TR.category).to_string()); print(pd.crosstab(TP.model, TP.category).to_string())

## Chequeo previo obligatorio: sensibilidad y VPP de `f1_iou05_resumen.csv` (±0,001); si falla, se detiene

In [ ]:
F1 = pd.read_csv(find('f1_iou05_resumen.csv')).set_index('modelo')
chk = []
for m in MODELS:
    r, p = TR[TR.model == LABEL[m]], TP[TP.model == LABEL[m]]
    se = (r.category == 'identified').mean(); pp = (p.category == 'tp').mean()
    nwn, ncp = int((r.category == 'wrong_number').sum()), int((p.category == 'wrong_number_counterpart').sum())
    chk.append(dict(modelo=LABEL[m], n_ref=len(r), sens=se, sens_ref=F1.loc[LABEL[m], 'sens'], ppv=pp, ppv_ref=F1.loc[LABEL[m], 'ppv'],
                    numero_equivocado=nwn, contrapartes=ncp))
chk = pd.DataFrame(chk)
chk['ok'] = ((chk.sens - chk.sens_ref).abs() <= TOL) & ((chk.ppv - chk.ppv_ref).abs() <= TOL) & (chk.n_ref == N_REF) & (chk.numero_equivocado == chk.contrapartes)
chk.to_csv(f'{OUT}/errores_v3_chequeo.csv', index=False); print(chk.round(4).to_string(index=False))
assert chk.ok.all(), 'EL CHEQUEO PREVIO FALLA (sensibilidad/VPP de f1_iou05_resumen.csv, n de referencia o contrapartes): se detiene'
print('chequeo OK')

## 1. Tabla principal por modelo, con IC 95 % por bootstrap de niños

In [ ]:
KIDS = np.array(sorted(man.child_id.unique())); KI = {c: i for i, c in enumerate(KIDS)}
rng = np.random.default_rng(0); draws = [rng.choice(KIDS, len(KIDS)) for _ in range(B)]
WK = np.stack([np.bincount([KI[c] for c in d], minlength=len(KIDS)) for d in draws]).astype(float)
def per_kid(d, cats):
    return pd.crosstab(d.child_id, d.category).reindex(index=KIDS, columns=cats, fill_value=0)
N_IMG = len(man)
summ = []
for m in MODELS:
    r, p = TR[TR.model == LABEL[m]], TP[TP.model == LABEL[m]]
    kr = per_kid(r, CAT_REF); n = kr.sum(1).values; bn = WK @ n
    row = dict(model=LABEL[m], n_ref=len(r), n_img=N_IMG, n_pred_classes=len(p), ppv=(p.category == 'tp').mean())
    for c in CAT_REF:
        row[f'pct_{c}'] = 100 * (r.category == c).mean(); bs = 100 * (WK @ kr[c].values) / bn
        row[f'pct_{c}_lo'], row[f'pct_{c}_hi'] = np.quantile(bs, .025), np.quantile(bs, .975); row[f'n_{c}'] = int((r.category == c).sum())
    for c in CAT_PRED:
        row[f'n_pred_{c}'] = int((p.category == c).sum()); row[f'pred_{c}_per_img'] = row[f'n_pred_{c}'] / N_IMG
    row['pred_unidentified_per_img'] = sum(row[f'n_pred_{c}'] for c in CAT_PRED) / N_IMG
    summ.append(row)
summ = pd.DataFrame(summ); summ.to_csv(f'{OUT}/errores_v3_resumen.csv', index=False)
assert np.allclose(summ[[f'pct_{c}' for c in CAT_REF]].sum(1), 100)
for _, r in summ.iterrows():
    print(f"\n{r.model} (n ref {r.n_ref}; clases predichas {r.n_pred_classes}; VPP {r.ppv:.4f})")
    for c in CAT_REF: print(f"   {c:22s} {r[f'pct_{c}']:6.2f} % [{r[f'pct_{c}_lo']:.2f}–{r[f'pct_{c}_hi']:.2f}]  n={r[f'n_{c}']}")
    print('   clases predichas no identificadas por foto: ' + ', '.join(f"{c} {r[f'pred_{c}_per_img']:.3f} (n={r[f'n_pred_{c}']})" for c in CAT_PRED) + f" | total {r.pred_unidentified_per_img:.3f}")

## 2. Números equivocados: pares más frecuentes y tipo de error

In [ ]:
def tipo(a, b):
    qa, qb, ua, ub = a // 10, b // 10, a % 10, b % 10
    if qa == qb: return 'neighbour, same quadrant' if abs(ua - ub) == 1 else 'same quadrant, not adjacent'
    if {qa, qb} in ({1, 2}, {3, 4}): return 'left-right, same arch'
    return 'upper-lower'
WN = TR[TR.category == 'wrong_number'].copy(); WN['pred_tooth'] = WN.pred_tooth.astype(int); WN['type'] = [tipo(a, b) for a, b in zip(WN.tooth, WN.pred_tooth)]
pairs, types, shifts = [], [], []
for m in MODELS:
    d = WN[WN.model == LABEL[m]]
    vc = d.groupby(['tooth', 'pred_tooth']).size().sort_values(ascending=False)
    pairs += [dict(model=LABEL[m], rank=i + 1, reference=a, predicted=b, n=int(n), n_mixed=int(((d.tooth == a) & (d.pred_tooth == b) & d.mixed).sum())) for i, ((a, b), n) in enumerate(vc.head(10).items())]
    t = d.type.value_counts(); types += [dict(model=LABEL[m], type=k, n=int(t.get(k, 0)), pct=100 * t.get(k, 0) / max(1, len(d)), n_total=len(d))
                                         for k in ['neighbour, same quadrant', 'same quadrant, not adjacent', 'left-right, same arch', 'upper-lower']]
    for a, b in ((6, 7), (6, 5), (7, 6)):
        s = d[(d.tooth % 10 == a) & (d.pred_tooth % 10 == b) & (d.tooth // 10 == d.pred_tooth // 10)]
        shifts.append(dict(model=LABEL[m], shift=f'{a}->{b}', n=len(s), n_mixed=int(s.mixed.sum()), detail=s.groupby(['tooth', 'pred_tooth']).size().to_dict().__repr__()))
pairs, types, shifts = pd.DataFrame(pairs), pd.DataFrame(types), pd.DataFrame(shifts)
pd.concat({'top_pairs': pairs, 'types': types, 'molar_shifts': shifts}, names=['table']).to_csv(f'{OUT}/errores_v3_numero_equivocado.csv')
for m in PIX_MODELS:
    print(f'\n{LABEL[m]}: {int((WN.model == LABEL[m]).sum())} números equivocados')
    print(pairs[pairs.model == LABEL[m]].drop(columns='model').to_string(index=False))
    print(types[types.model == LABEL[m]].drop(columns='model').round(1).to_string(index=False))
    print(shifts[shifts.model == LABEL[m]].drop(columns='model').to_string(index=False))
print('\nTipos, todos los modelos:'); print(types.pivot(index='type', columns='model', values='pct').round(1).to_string())

## 3. Subgrupos: vista, retractor y dentición mixta contra el resto

In [ ]:
def block(r, p, name, val):
    o = dict(group=name, value=val, n_img=r.image_id.nunique(), n_children=r.child_id.nunique(), n_ref=len(r))
    for c in CAT_REF: o[f'pct_{c}'] = 100 * (r.category == c).mean()
    nimg = r.image_id.nunique()
    for c in CAT_PRED: o[f'pred_{c}_per_img'] = (p.category == c).sum() / nimg
    o['pred_unidentified_per_img'] = p.category.isin(CAT_PRED).sum() / nimg
    return o
sub = []
for m in MODELS:
    r0, p0 = TR[TR.model == LABEL[m]], TP[TP.model == LABEL[m]]
    for col, name in (('view', 'view'), ('condition', 'retractor'), ('mixed', 'mixed dentition')):
        for v in sorted(r0[col].unique(), key=str):
            sub.append(dict(model=LABEL[m], **block(r0[r0[col] == v], p0[p0[col] == v], name, v)))
sub = pd.DataFrame(sub); sub.to_csv(f'{OUT}/errores_v3_subgrupos.csv', index=False)
for m in PIX_MODELS:
    print(f'\n{LABEL[m]}'); print(sub[sub.model == LABEL[m]].drop(columns='model').round(2).to_string(index=False))

## 4. Dentición mixta: clases predichas sobre dientes temporales y molares 6
Cruce píxel a píxel con las predicciones `fold<k>_preds.npz` del benchmark, que están a la resolución nativa de la máscara (se verifica la forma; no se reescala). Para cada clase predicha \"sobre fondo\" en las fotos mixtas: fracción de sus píxeles sobre un diente temporal; ≥ 50 % → \"sobre temporal\".

In [ ]:
RES = os.path.dirname(find('metricas_ic95.csv'))
mix_ids = set(man[man.mixed].image_id)
temp_rows, cover_rows, MISMATCH = [], [], []
for m in PIX_MODELS:
    for k in range(5):
        Pz = np.load(f'{RES}/{m}/fold{k}_preds.npz')
        for j in man[(man.fold == k) & man.mixed].image_id:
            n_prim, packed, shp, _ = INFO[j]; pred = Pz[str(j)]
            assert pred.shape == shp, f'{m} {j}: forma {pred.shape} distinta de la máscara {shp}'
            prim = np.unpackbits(packed)[:shp[0] * shp[1]].reshape(shp).astype(bool)
            cover_rows.append(dict(model=LABEL[m], image_id=j, primary_px=int(prim.sum()), primary_px_pred_as_tooth=int((pred[prim] > 0).sum())))
            sel = TP[(TP.model == LABEL[m]) & (TP.image_id == j) & (TP.category == 'on_background')]
            for t, npx in zip(sel.tooth, sel.pred_px):
                c = FDI.index(t); pm = pred == c
                if int(pm.sum()) != npx: MISMATCH.append((m, j, t, int(pm.sum()), npx))
                temp_rows.append(dict(model=LABEL[m], image_id=j, tooth=t, pred_px=npx, frac_on_primary=(pm & prim).sum() / npx))
        del Pz
print('clases con píxeles distintos entre preds.npz y el caché:', len(MISMATCH), MISMATCH[:5])
TEMP = pd.DataFrame(temp_rows); TEMP['on_primary'] = TEMP.frac_on_primary >= 0.5
COV = pd.DataFrame(cover_rows)
mixres = []
for m in PIX_MODELS:
    d = TEMP[TEMP.model == LABEL[m]]; pm_all = TP[(TP.model == LABEL[m]) & TP.mixed]
    cv = COV[COV.model == LABEL[m]]
    mixres.append(dict(model=LABEL[m], n_img_mixed=len(mix_ids), pred_unidentified_mixed=int(pm_all.category.isin(CAT_PRED).sum()),
                       on_background_mixed=len(d), on_primary=int(d.on_primary.sum()),
                       pct_primary_px_pred_as_tooth=100 * cv.primary_px_pred_as_tooth.sum() / cv.primary_px.sum(),
                       fdi_on_primary=d[d.on_primary].tooth.value_counts().sort_index().to_dict().__repr__()))
mixres = pd.DataFrame(mixres)
m6 = []
for m in MODELS:
    r = TR[(TR.model == LABEL[m]) & (TR.tooth % 10 == 6)]
    for v, d in r.groupby('mixed'):
        m6.append(dict(model=LABEL[m], mixed=v, n_ref=len(d), **{f'pct_{c}': 100 * (d.category == c).mean() for c in CAT_REF},
                       shift_6_to_7=int(((d.category == 'wrong_number') & (d.pred_tooth % 10 == 7)).sum())))
m6 = pd.DataFrame(m6)
pd.concat({'mixed_summary': mixres, 'first_molars': m6}, names=['table']).to_csv(f'{OUT}/errores_v3_denticion_mixta.csv')
TEMP.to_csv(f'{OUT}/errores_v3_mixta_clases_sobre_fondo.csv', index=False)
print(mixres.round(2).to_string(index=False)); print(m6.round(1).to_string(index=False))
for m in PIX_MODELS:
    d = TEMP[(TEMP.model == LABEL[m]) & TEMP.on_primary]
    print(f'{LABEL[m]}: FDI recibido por las {len(d)} clases sobre temporales:', d.tooth.value_counts().sort_index().to_dict())

## 5. Por niño (SegFormer-B2): % identificados

In [ ]:
pk = []
for m in MODELS:
    r = TR[TR.model == LABEL[m]]; g = r.groupby('child_id').category.apply(lambda s: 100 * (s == 'identified').mean())
    pk.append(dict(model=LABEL[m], n_children=len(g), median=g.median(), p10=g.quantile(.10), p25=g.quantile(.25), minimum=g.min(),
                   n_below_90=int((g < 90).sum()), n_below_80=int((g < 80).sum()),
                   child_min=g.idxmin(), child_min_mixed=bool(man[man.child_id == g.idxmin()].mixed.any())))
pk = pd.DataFrame(pk); pk.to_csv(f'{OUT}/errores_v3_por_nino.csv', index=False); print(pk.round(1).to_string(index=False))

## Lectura automática (para el informe)

In [ ]:
s = summ.set_index('model')
for m in ('SegFormer-B2', 'YOLO11m-seg'):
    r = s.loc[m]
    print(f"{m}: de {int(r.n_ref)} dientes de referencia, identificados {r.pct_identified:.1f} % [{r.pct_identified_lo:.1f}–{r.pct_identified_hi:.1f}]; "
          f"número equivocado {r.pct_wrong_number:.1f} %; contorno insuficiente {r.pct_insufficient_contour:.1f} %; no detectados {r.pct_not_detected:.1f} %; "
          f"clases predichas no identificadas {r.pred_unidentified_per_img:.2f}/foto (sobre fondo {r.pred_on_background_per_img:.2f}).")
t = types.set_index(['model', 'type']).pct
print(f"Vecino del mismo cuadrante: SegFormer-B2 {t[('SegFormer-B2', 'neighbour, same quadrant')]:.0f} %, YOLO11m-seg {t[('YOLO11m-seg', 'neighbour, same quadrant')]:.0f} %")
sm = sub[(sub.group == 'mixed dentition')].set_index(['model', 'value']).pct_identified
print(f"Identificados mixta vs resto: SegFormer-B2 {sm[('SegFormer-B2', True)]:.1f} % vs {sm[('SegFormer-B2', False)]:.1f} %; YOLO11m-seg {sm[('YOLO11m-seg', True)]:.1f} % vs {sm[('YOLO11m-seg', False)]:.1f} %")
print('\ntiempo total', round((time.time() - T0) / 60, 1), 'min'); print(sorted(os.listdir(OUT)))